# SAS → Databricks Conversion Pipeline
## Step-by-Step Test Notebook

This notebook replicates the full backend pipeline so you can test and inspect each stage independently.

**Pipeline stages:**
1. ⚙️ Setup & Configuration
2. 📂 Load SAS File
3. 🔍 Classify SAS Complexity (LLM)
4. 🗺️ Source Mapping (manual inputs)
5. 🔧 Macro Resolution (manual inputs)
6. 📋 Plan Generation + Validation Loop (LLM)
7. ✏️ Plan Review (inspect & edit)
8. 💻 Code Generation + Per-Cell Validation (LLM)
9. 📥 Build & Save Output Notebook

---
## Cell 1 — Setup & Configuration
Set your API key, choose models, and point to your SAS file.

In [ ]:
import json, re, time, os
import google.generativeai as genai
from google.generativeai import GenerationConfig
from pprint import pprint
from copy import deepcopy

# ─── CONFIGURE THESE ────────────────────────────────────────────────────────
API_KEY         = "YOUR_GEMINI_API_KEY_HERE"   # paste your key
ANALYSIS_MODEL  = "gemma-4-26b-a4b-it"         # Classifier, Planner, Code Gen
VALIDATION_MODEL = "gemini-3.1-flash-lite-preview"  # Plan Validator, Code Validator
SAS_FILE_PATH   = r"path\to\your\file.sas"    # path to the SAS file
OUTPUT_PATH     = r"databricks_output.py"      # where to save the generated notebook
MAX_PLAN_LOOPS  = 5
MAX_CELL_RETRIES = 5
# ────────────────────────────────────────────────────────────────────────────

print(f"✅ Config loaded")
print(f"   Analysis model  : {ANALYSIS_MODEL}")
print(f"   Validation model: {VALIDATION_MODEL}")
print(f"   SAS file        : {SAS_FILE_PATH}")

---
## Cell 2 — LLM Service Functions
Two functions matching the backend `services/llm.py`:
- `call_llm_validated` → JSON-mode (schema-enforced)
- `call_llm_freeform`  → plain text (code generation)

In [ ]:
def _get_client(api_key: str, model: str):
    genai.configure(api_key=api_key)
    return genai.GenerativeModel(model_name=model)


def call_llm_validated(prompt: str, api_key: str, model: str, retries: int = 3) -> dict:
    """
    Call Gemini with JSON output mode enforced.
    Used for: Complexity Classifier, Planning LLM, Plan Validator, Code Validator.
    """
    client = _get_client(api_key, model)
    config = GenerationConfig(response_mime_type="application/json", temperature=0.2)
    last_error = None
    for attempt in range(retries):
        try:
            response = client.generate_content(prompt, generation_config=config)
            text = response.text.strip()
            if text.startswith("```"):
                lines = text.split("\n")
                text = "\n".join(lines[1:-1]) if len(lines) > 2 else text
            return json.loads(text)
        except (json.JSONDecodeError, Exception) as e:
            last_error = e
            print(f"   ⚠️  call_llm_validated attempt {attempt+1} failed: {e}")
            if attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
    raise ValueError(f"call_llm_validated failed after {retries} retries: {last_error}")


def call_llm_freeform(prompt: str, api_key: str, model: str) -> str:
    """
    Call Gemini for free-form text (Python code) output.
    Used for: Code Generation cells.
    """
    client = _get_client(api_key, model)
    config = GenerationConfig(temperature=0.3)
    response = client.generate_content(prompt, generation_config=config)
    text = response.text.strip()
    if text.startswith("```"):
        lines = text.split("\n")
        if len(lines) > 2:
            text = "\n".join(lines[1:-1])
    return text


print("✅ LLM service functions defined")
print("   call_llm_validated → JSON mode (Classifier / Validators)")
print("   call_llm_freeform  → Plain text (Code Gen)")

---
## Cell 3 — LLM Prompts
All five prompt templates from `backend/prompts/`.

In [ ]:
CLASSIFIER_PROMPT = """\
You are a SAS-to-Databricks complexity classifier.

Analyse the SAS code provided and return a JSON object with EXACTLY this structure:
{{
  "complexity_level": "low" | "medium" | "high",
  "complexity_score": <integer 1-10>,
  "complexity_reasoning": "<one paragraph explaining the score>",
  "libnames": [
    {{"alias": "<libname alias>", "source_hint": "<detected source system e.g. Netezza, Oracle, SAS dataset>"}}
  ],
  "procs": ["<list of SAS PROC types found>"],
  "macros": [
    {{
      "name": "<macro name including %>",
      "complexity": "simple" | "complex",
      "is_flagged": true | false,
      "flag_reason": "<reason or null>"
    }}
  ],
  "cross_source_joins": true | false,
  "flagged_constructs": [
    {{"construct": "<name>", "lines": "<range>", "reason": "<why flagged>"}}
  ]
}}

Flag: %include, PROC FORMAT, PROC TRANSPOSE, PROC MEANS/SUMMARY, PROC REPORT,
nested macros (depth>2), SAS date functions, any macro judged too complex.

Return ONLY the JSON. No prose, no markdown fences.

SAS CODE:
{sas_code}
"""

PREVIOUS_ATTEMPTS_TEMPLATE = """
PREVIOUS PLANNING ATTEMPTS (you have failed {n} times — study your mistakes):
{issues_summary}
Ensure you explicitly address each blocking issue listed above.
"""

PLANNER_PROMPT = """\
You are a SAS-to-Databricks planning expert. Create a step-by-step execution plan.

Return ONLY a JSON object with EXACTLY this structure:
{{
  "plan_steps": [
    {{
      "step_id": <int>,
      "step_type": "extract_stage" | "join" | "transform" | "output",
      "sas_source_block": "<reference e.g. PROC SQL block 1>",
      "instruction": "<precise English instruction for code generation>",
      "source_libname": "<libname alias or null>",
      "target_table": "<fully qualified table name or null>",
      "cross_source": true | false,
      "flagged": false,
      "flag_reason": null,
      "edited_by_user": false
    }}
  ],
  "planning_reasoning": "<chain-of-thought explaining your plan>"
}}

CRITICAL RULES:
- Every cross-source join MUST produce TWO steps: extract_stage then join.
- extract_stage target_table must be: <catalog>.<schema>_stage.<table>_stage
- Instructions must be fully self-contained — no SAS syntax.
- Steps in execution order.

SAS CODE:
{sas_code}

CLASSIFIER OUTPUT:
{classifier_output}

TRANSLATION MANIFEST:
{manifest}

MACRO RESOLUTIONS:
{macro_resolutions}

{previous_attempts_summary}
"""

PLAN_VALIDATOR_PROMPT = """\
You are a senior SAS-to-Databricks plan validator.

Return ONLY a JSON object with EXACTLY this structure:
{{
  "validation_pass": true | false,
  "issues": [
    {{"step_id": <int or null>, "severity": "blocking" | "warning", "issue": "<description>"}}
  ]
}}

VALIDATION RULES:
1. Every cross-source join must have a preceding extract_stage step.
2. Staging table names must be referenced exactly in the join step.
3. All libname aliases in the SAS code have corresponding steps.
4. Macro instructions are reflected in step instructions.
5. Steps in logical execution order.
6. No SAS syntax in any instruction.
7. No table spanning two libname aliases without a preceding extract_stage.

Set validation_pass to true only if there are NO blocking issues.

PROPOSED PLAN:
{plan_output}

ORIGINAL SAS CODE:
{sas_code}
"""

CODE_GEN_PROMPT = """\
You are a SAS-to-Databricks code generation expert. Generate ONE Databricks Python notebook cell.

OUTPUT FORMAT — return ONLY Python code (no markdown fences, no explanation).
Start with this exact header:
# COMMAND ----------
# Step {step_id} of {total_steps} | Type: {step_type}
# Instruction: {instruction_truncated}
# SAS Source: {sas_source_block}
# Validator: Approved after {{N}} loop(s)
# Flag: None

Then the PySpark code implementing the instruction.

RULES:
- Use PySpark (spark.read, spark.sql, DataFrame API).
- extract_stage: spark.read.format("jdbc") → write to Delta as saveAsTable.
- Credential values use angle-bracket placeholders: <JDBC_URL>, <USERNAME>, <PASSWORD>.
- Staging table name in Cell A MUST exactly match what Cell B references.
- Do NOT include prose — only the header comment block + Python code.

STEP INFO:
Step ID       : {step_id}
Total Steps   : {total_steps}
Step Type     : {step_type}
Instruction   : {instruction}
SAS Source Ref: {sas_source_block}
Target Table  : {target_table}

SAS SOURCE CODE BLOCK:
{sas_block}

{retry_context}
"""

RETRY_CONTEXT_TEMPLATE = """
PREVIOUS ATTEMPT FAILED — Code Validator issues (attempt {attempt} of 5):
{issues}
Fix all listed issues. Do not repeat the same mistakes.
"""

CODE_VALIDATOR_PROMPT = """\
You are a Databricks code validator.

Return ONLY a JSON object with EXACTLY this structure:
{{
  "cell_pass": true | false,
  "issues": [
    {{"severity": "blocking" | "warning", "issue": "<description>"}}
  ],
  "validator_reasoning": "<brief explanation>"
}}

CHECKS:
1. Cell starts with # COMMAND ---------- header.
2. Step type matches code pattern.
3. extract_stage: staging table name in write matches target_table exactly.
4. join: references the exact staging table from extract_stage.
5. No SAS syntax in generated Python.
6. JDBC credentials use angle-bracket placeholders.
7. Instruction intent is fully satisfied.

INSTRUCTION:
{instruction}

TARGET TABLE:
{target_table}

GENERATED CELL CODE:
{cell_code}
"""

print("✅ All 5 LLM prompt templates defined")
print("   CLASSIFIER_PROMPT, PLANNER_PROMPT, PLAN_VALIDATOR_PROMPT")
print("   CODE_GEN_PROMPT, CODE_VALIDATOR_PROMPT")

---
## Cell 4 — Load SAS File
Reads the SAS file from disk.

In [ ]:
with open(SAS_FILE_PATH, "r", encoding="utf-8", errors="replace") as f:
    SAS_CODE = f.read()

print(f"✅ Loaded SAS file: {SAS_FILE_PATH}")
print(f"   Lines : {len(SAS_CODE.splitlines())}")
print(f"   Chars : {len(SAS_CODE):,}")
print("")
print("─" * 60)
print("FIRST 40 LINES:")
print("─" * 60)
print("\n".join(SAS_CODE.splitlines()[:40]))

---
## Cell 5 — Stage 1: Complexity Classification
Sends the SAS code to the Classifier LLM (`call_llm_validated`).
Returns `ClassifierOutput` with complexity level, libnames, macros, flagged constructs.

In [ ]:
print("🔍 Running Complexity Classifier LLM...")
print(f"   Model: {ANALYSIS_MODEL}")
print("")

classifier_prompt = CLASSIFIER_PROMPT.format(sas_code=SAS_CODE)
CLASSIFIER_OUTPUT = call_llm_validated(classifier_prompt, API_KEY, ANALYSIS_MODEL)

print("─" * 60)
print("CLASSIFIER OUTPUT")
print("─" * 60)
print(f"  Complexity Level : {CLASSIFIER_OUTPUT.get('complexity_level', '?').upper()}")
print(f"  Complexity Score : {CLASSIFIER_OUTPUT.get('complexity_score', '?')}/10")
print(f"  Reasoning        : {CLASSIFIER_OUTPUT.get('complexity_reasoning', '')[:200]}...")
print()

libnames = CLASSIFIER_OUTPUT.get('libnames', [])
print(f"  Libnames ({len(libnames)}):")
for lb in libnames:
    print(f"    {lb['alias']:20s}  ← {lb['source_hint']}")

procs = CLASSIFIER_OUTPUT.get('procs', [])
print(f"\n  PROCs: {', '.join(procs) if procs else 'none'}")

macros = CLASSIFIER_OUTPUT.get('macros', [])
print(f"\n  Macros ({len(macros)}):")
for m in macros:
    flag = "  ⚠️  FLAGGED" if m.get('is_flagged') else ""
    print(f"    {m['name']:30s}  [{m['complexity']}]{flag}")
    if m.get('flag_reason'):
        print(f"      reason: {m['flag_reason']}")

flagged = CLASSIFIER_OUTPUT.get('flagged_constructs', [])
if flagged:
    print(f"\n  ⚠️  Flagged Constructs ({len(flagged)}):")
    for fc in flagged:
        print(f"    {fc['construct']} (lines {fc['lines']}): {fc['reason']}")

cross = CLASSIFIER_OUTPUT.get('cross_source_joins', False)
print(f"\n  Cross-source joins detected: {'YES ⚠️' if cross else 'NO'}")
print("\n✅ Classification complete")

---
## Cell 6 — Stage 2: Source Mapping (manual)
For each detected libname, provide the Databricks target catalog, schema, and source JDBC type.

Edit the `LIBNAME_MAPPINGS` dict below, then run the cell.

In [ ]:
# ─── EDIT THIS: one entry per libname detected above ─────────────────────────
# Formats:
#   keep_original=True  → Databricks Delta table, no JDBC needed
#   keep_original=False → external source, JDBC connection will be generated

LIBNAME_MAPPINGS = [
    # Example — fill in one dict per libname:
    {
        "alias": "LIBNAME_ALIAS",          # must match classifier output
        "original_source": "Netezza",      # from classifier
        "target_catalog": "prod",           # Databricks catalog
        "target_schema": "analytics",       # Databricks schema
        "jdbc_type": "netezza",             # netezza | oracle | sqlserver | postgresql | databricks | other
        "keep_original": False,
    },
    # Add more entries here for each libname...
]

# ─────────────────────────────────────────────────────────────────────────────

TRANSLATION_MANIFEST = {"libname_mappings": LIBNAME_MAPPINGS}

print("✅ Source mapping defined")
print(f"   {len(LIBNAME_MAPPINGS)} libname(s) mapped:")
for m in LIBNAME_MAPPINGS:
    prefix = "  🔗 JDBC" if not m["keep_original"] else "  📁 Native"
    print(f"  {prefix}  {m['alias']:20s} → {m['target_catalog']}.{m['target_schema']}  [{m['jdbc_type']}]")

---
## Cell 7 — Stage 3: Macro Resolution (manual)
For each macro detected by the Classifier, choose:
- `skip` — ignore this macro
- `user_instruction` — provide a plain-English instruction for the planner
- `stop_workflow` — abort (do not use, left for completeness)

In [ ]:
# ─── EDIT THIS: one entry per macro detected above ──────────────────────────
MACRO_RESOLUTIONS = [
    # Example:
    # {
    #     "macro_name": "%get_region",
    #     "resolution_type": "user_instruction",
    #     "instruction": "Apply WHERE region = 'WEST' to the query",
    #     "stop_workflow": False,
    # },
    # {
    #     "macro_name": "%load_dates",
    #     "resolution_type": "skip",
    #     "instruction": None,
    #     "stop_workflow": False,
    # },
]

# Auto-detect macros from classifier and default to 'skip' if not listed above
resolved_names = {r["macro_name"] for r in MACRO_RESOLUTIONS}
for macro in CLASSIFIER_OUTPUT.get("macros", []):
    if macro["name"] not in resolved_names:
        MACRO_RESOLUTIONS.append({
            "macro_name": macro["name"],
            "resolution_type": "skip",
            "instruction": None,
            "stop_workflow": False,
        })
        print(f"  ⚙️  Auto-defaulted {macro['name']} → skip")

MACRO_RESOLUTIONS_OBJ = {"resolutions": MACRO_RESOLUTIONS}

print("\n✅ Macro resolutions:")
for r in MACRO_RESOLUTIONS:
    tag = f"→ '{r['instruction']}'" if r["resolution_type"] == "user_instruction" else ""
    print(f"   {r['macro_name']:30s}  [{r['resolution_type']}] {tag}")

---
## Cell 8 — Stage 4+5: Plan Generation + Validation Loop
Runs the Planning LLM → Plan Validator loop (up to `MAX_PLAN_LOOPS` iterations).
Prints each loop's result and blocking issues.

In [ ]:
def build_planner_prompt(sas_code, classifier_output, manifest, macro_resolutions, previous_issues=None):
    prev_summary = ""
    if previous_issues:
        lines = []
        for i, attempt in enumerate(previous_issues, 1):
            for issue in attempt.get("issues", []):
                lines.append(f"  Attempt {i}: [{issue.get('severity','?')}] step {issue.get('step_id','?')} — {issue.get('issue','?')}")
        prev_summary = PREVIOUS_ATTEMPTS_TEMPLATE.format(
            n=len(previous_issues),
            issues_summary="\n".join(lines)
        )
    return PLANNER_PROMPT.format(
        sas_code=sas_code,
        classifier_output=json.dumps(classifier_output, indent=2),
        manifest=json.dumps(manifest, indent=2),
        macro_resolutions=json.dumps(macro_resolutions, indent=2),
        previous_attempts_summary=prev_summary,
    )


print("📋 Starting Plan Generation + Validation loop...")
print(f"   Planning model  : {ANALYSIS_MODEL}")
print(f"   Validator model : {VALIDATION_MODEL}")
print(f"   Max loops       : {MAX_PLAN_LOOPS}")
print()

LOOP_TRACE = []
previous_issues = []
PLAN_OUTPUT = None
LAST_VALIDATION = None
PLAN_STATUS = "pending"

for loop_num in range(1, MAX_PLAN_LOOPS + 1):
    print(f"{'─'*60}")
    print(f"  LOOP {loop_num} of {MAX_PLAN_LOOPS}")
    print(f"{'─'*60}")

    # --- Planning LLM ---
    print(f"  ⏳ Calling Planning LLM ({ANALYSIS_MODEL})...")
    plan_prompt = build_planner_prompt(
        SAS_CODE, CLASSIFIER_OUTPUT, TRANSLATION_MANIFEST,
        MACRO_RESOLUTIONS_OBJ,
        previous_issues=previous_issues if loop_num > 1 else None
    )
    plan_result = call_llm_validated(plan_prompt, API_KEY, ANALYSIS_MODEL)

    steps = plan_result.get("plan_steps", [])
    reasoning = plan_result.get("planning_reasoning", "")[:300]
    print(f"  ✅ Plan generated: {len(steps)} step(s)")
    print(f"  📝 Reasoning: {reasoning}...")
    print()
    for s in steps:
        cross = "[cross-source]" if s.get("cross_source") else ""
        print(f"    Step {s['step_id']:2d}  [{s['step_type']:15s}] {cross}")
        print(f"           {s['instruction'][:100]}")

    # --- Plan Validator LLM ---
    print(f"\n  ⏳ Calling Plan Validator ({VALIDATION_MODEL})...")
    val_prompt = PLAN_VALIDATOR_PROMPT.format(
        plan_output=json.dumps(plan_result, indent=2),
        sas_code=SAS_CODE,
    )
    val_result = call_llm_validated(val_prompt, API_KEY, VALIDATION_MODEL)

    passed = val_result.get("validation_pass", False)
    issues = val_result.get("issues", [])
    blocking = [i for i in issues if i.get("severity") == "blocking"]
    warnings = [i for i in issues if i.get("severity") == "warning"]

    status_icon = "✅" if passed else "❌"
    print(f"  {status_icon} Validation {'PASSED' if passed else 'FAILED'}  "
          f"| {len(blocking)} blocking, {len(warnings)} warnings")

    if blocking:
        print(f"  ⛔ Blocking issues:")
        for issue in blocking:
            print(f"     Step {issue.get('step_id','?')}: {issue.get('issue','?')}")

    if warnings:
        print(f"  ⚠️  Warnings:")
        for issue in warnings:
            print(f"     Step {issue.get('step_id','?')}: {issue.get('issue','?')}")

    LOOP_TRACE.append({"loop": loop_num, "plan": plan_result, "validation": val_result})

    if passed or not blocking:
        PLAN_OUTPUT = plan_result
        LAST_VALIDATION = val_result
        PLAN_STATUS = "passed"
        print(f"\n✅ Plan passed validation on loop {loop_num}!")
        break

    previous_issues.append(val_result)

else:
    # Max loops reached without passing
    last = LOOP_TRACE[-1]
    PLAN_OUTPUT = last["plan"]
    LAST_VALIDATION = last["validation"]
    PLAN_STATUS = "max_loops_reached"
    print(f"\n⚠️  Max loops ({MAX_PLAN_LOOPS}) reached without full validation pass.")
    print("   Proceeding with last plan — review blocking issues above before continuing.")

print(f"\n📊 Loop summary: {len(LOOP_TRACE)} loop(s) run | Status: {PLAN_STATUS}")
print(f"   Final plan has {len(PLAN_OUTPUT.get('plan_steps', []))} steps")

---
## Cell 9 — Stage 6: Plan Review & Edit
Inspect the plan. Edit any `instruction` text directly and re-run this cell.
Call `run_final_validation()` after editing to confirm no new issues.

In [ ]:
import copy

# Work on a mutable copy
APPROVED_PLAN = copy.deepcopy(PLAN_OUTPUT)
plan_steps = APPROVED_PLAN["plan_steps"]

print("📋 PLAN REVIEW")
print("=" * 70)
for step in plan_steps:
    edited = " [EDITED]" if step.get("edited_by_user") else ""
    flagged = " [FLAGGED]" if step.get("flagged") else ""
    cross = " [cross-source]" if step.get("cross_source") else ""
    print(f"\nStep {step['step_id']:2d} | {step['step_type']:15s}{cross}{edited}{flagged}")
    print(f"  SAS ref    : {step['sas_source_block']}")
    print(f"  Instruction: {step['instruction']}")
    if step.get("target_table"):
        print(f"  Target     : {step['target_table']}")

print("\n" + "=" * 70)
print("\nTo edit a step's instruction, use the function below:")
print("  edit_step(step_id=2, new_instruction='...')")
print("Then run run_final_validation() to confirm before proceeding.")


def edit_step(step_id: int, new_instruction: str):
    for step in plan_steps:
        if step["step_id"] == step_id:
            old = step["instruction"]
            step["instruction"] = new_instruction
            step["edited_by_user"] = True
            print(f"✅ Step {step_id} instruction updated")
            print(f"   OLD: {old[:120]}")
            print(f"   NEW: {new_instruction[:120]}")
            return
    print(f"❌ Step {step_id} not found")


def run_final_validation():
    print(f"\n⏳ Running final Plan Validator pass ({VALIDATION_MODEL})...")
    val_prompt = PLAN_VALIDATOR_PROMPT.format(
        plan_output=json.dumps(APPROVED_PLAN, indent=2),
        sas_code=SAS_CODE,
    )
    result = call_llm_validated(val_prompt, API_KEY, VALIDATION_MODEL)
    passed = result.get("validation_pass", False)
    issues = result.get("issues", [])
    blocking = [i for i in issues if i.get("severity") == "blocking"]
    icon = "✅" if passed else "❌"
    print(f"{icon} Final validation {'PASSED' if passed else 'FAILED'}  "
          f"| {len(blocking)} blocking issue(s)")
    if blocking:
        for issue in blocking:
            print(f"  ⛔ Step {issue.get('step_id','?')}: {issue.get('issue','?')}")
    else:
        print("  Plan is approved — safe to proceed to code generation.")
    return result


# Example (uncomment and adjust):
# edit_step(step_id=2, new_instruction="Left join member_dim_stage with claims_fact on member_id")
# run_final_validation()

---
## Cell 10 — Stage 7: Code Generation + Per-Cell Validation
Generates each notebook cell using `call_llm_freeform`, then validates with `call_llm_validated`.
Retries up to `MAX_CELL_RETRIES` per cell; marks failures as `FLAGGED`.

In [ ]:
print("💻 Starting Code Generation...")
print(f"   Code Gen model      : {ANALYSIS_MODEL}")
print(f"   Code Validator model: {VALIDATION_MODEL}")
print(f"   Max retries / cell  : {MAX_CELL_RETRIES}")
print()

total_steps = len(APPROVED_PLAN["plan_steps"])
GENERATED_CELLS = []

for step in APPROVED_PLAN["plan_steps"]:
    print(f"{'─'*60}")
    print(f"  Step {step['step_id']} of {total_steps} | [{step['step_type']}]")
    print(f"  {step['instruction'][:100]}...")

    cell_code = None
    final_status = "flagged"
    flag_reason = None
    retry_context = ""

    for attempt in range(1, MAX_CELL_RETRIES + 1):
        print(f"\n  ⏳ Attempt {attempt}/{MAX_CELL_RETRIES} — generating code ({ANALYSIS_MODEL})...")

        # --- Code Gen (freeform) ---
        gen_prompt = CODE_GEN_PROMPT.format(
            step_id=step["step_id"],
            total_steps=total_steps,
            step_type=step["step_type"],
            instruction=step["instruction"],
            instruction_truncated=step["instruction"][:200],
            sas_source_block=step["sas_source_block"],
            target_table=step.get("target_table") or "N/A",
            sas_block=SAS_CODE,
            retry_context=retry_context,
        )
        try:
            generated = call_llm_freeform(gen_prompt, API_KEY, ANALYSIS_MODEL)
        except Exception as e:
            print(f"  ❌ Code generation failed: {e}")
            flag_reason = str(e)
            break

        print(f"  📝 Generated {len(generated.splitlines())} lines of code")

        # --- Code Validator (validated) ---
        print(f"  ⏳ Validating code ({VALIDATION_MODEL})...")
        val_prompt = CODE_VALIDATOR_PROMPT.format(
            instruction=step["instruction"],
            target_table=step.get("target_table") or "N/A",
            cell_code=generated,
        )
        try:
            val_result = call_llm_validated(val_prompt, API_KEY, VALIDATION_MODEL)
        except Exception as e:
            print(f"  ❌ Validation call failed: {e}")
            flag_reason = str(e)
            cell_code = generated
            break

        cell_passed = val_result.get("cell_pass", False)
        val_issues = val_result.get("issues", [])
        blocking = [i for i in val_issues if i.get("severity") == "blocking"]
        print(f"  {'✅' if cell_passed else '❌'} Validation {'PASSED' if cell_passed else 'FAILED'} "
              f"| {len(blocking)} blocking")
        print(f"     Reasoning: {val_result.get('validator_reasoning','')[:120]}")

        if cell_passed:
            cell_code = generated.replace("Approved after {N} loop(s)", f"Approved after {attempt} loop(s)")
            final_status = "approved"
            print(f"  ✅ APPROVED on attempt {attempt}")
            break
        else:
            issue_lines = "\n".join(f"  [{i.get('severity')}] {i.get('issue')}" for i in val_issues)
            print(f"  Issues:\n{issue_lines}")
            retry_context = RETRY_CONTEXT_TEMPLATE.format(attempt=attempt, issues=issue_lines)
            cell_code = generated
            flag_reason = issue_lines

    # If still flagged after retries
    if final_status == "flagged" and cell_code:
        cell_code = re.sub(r"# Validator:.*", "# Validator: FLAGGED - see below", cell_code)
        cell_code = re.sub(r"# Flag: None", f"# Flag: {flag_reason or 'Max retries exceeded'}", cell_code)
        print(f"  ⛔ FLAGGED after {MAX_CELL_RETRIES} attempts")

    GENERATED_CELLS.append({
        "step_id": step["step_id"],
        "status": final_status,
        "cell_code": cell_code or f"# COMMAND ----------\n# Step {step['step_id']} | FLAGGED\n# Flag: {flag_reason}",
        "flag_reason": flag_reason,
    })

# Summary
approved = [c for c in GENERATED_CELLS if c["status"] == "approved"]
flagged  = [c for c in GENERATED_CELLS if c["status"] == "flagged"]
print(f"\n{'='*60}")
print(f"✅ CODE GENERATION COMPLETE")
print(f"   Total cells  : {len(GENERATED_CELLS)}")
print(f"   Approved     : {len(approved)}")
print(f"   Flagged      : {len(flagged)}")
if flagged:
    print(f"   ⛔ Flagged steps: {[c['step_id'] for c in flagged]}")

---
## Cell 11 — Stage 8: Build & Save Output Notebook
Assembles all generated cells into a Databricks `.py` notebook file.

In [ ]:
def build_notebook(generated_cells, sas_filename="unknown.sas") -> str:
    flagged_cells = [c for c in generated_cells if c["status"] == "flagged"]
    approved_cells = [c for c in generated_cells if c["status"] == "approved"]

    run_summary = {
        "source_file": sas_filename,
        "total_plan_steps": len(generated_cells),
        "cells_approved": len(approved_cells),
        "cells_flagged": len(flagged_cells),
        "flagged_steps": [c["step_id"] for c in flagged_cells],
        "complexity": CLASSIFIER_OUTPUT.get("complexity_level"),
        "complexity_score": CLASSIFIER_OUTPUT.get("complexity_score"),
        "cross_source_joins": CLASSIFIER_OUTPUT.get("cross_source_joins"),
        "plan_loops_used": len(LOOP_TRACE),
        "plan_status": PLAN_STATUS,
    }

    header = (
        "# Databricks notebook source\n"
        "# Generated by SAS → Databricks Conversion Tool\n"
        "# \n"
        f"# RUN SUMMARY: {json.dumps(run_summary, indent=2).replace(chr(10), chr(10) + '# ')}\n"
        "# \n"
        "# Search for '<' to find all credential placeholders requiring values\n"
    )

    # Manifest summary section
    manifest_lines = ["# MANIFEST SUMMARY:"]
    for m in LIBNAME_MAPPINGS:
        arrow = f"{m['target_catalog']}.{m['target_schema']}" if not m["keep_original"] else "(keep original)"
        manifest_lines.append(f"#   {m['alias']} ({m['original_source']}) → {arrow}")
    header += "\n".join(manifest_lines) + "\n"

    body_parts = [header]
    for cell in generated_cells:
        body_parts.append(cell["cell_code"])

    return "\n\n".join(body_parts)


notebook_text = build_notebook(GENERATED_CELLS, sas_filename=os.path.basename(SAS_FILE_PATH))

# Save to file
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    f.write(notebook_text)

print(f"✅ Notebook saved to: {OUTPUT_PATH}")
print(f"   Total lines: {len(notebook_text.splitlines()):,}")
print(f"   Total chars: {len(notebook_text):,}")
print(f"   Cells: {len(GENERATED_CELLS)} total | "
      f"{len([c for c in GENERATED_CELLS if c['status']=='approved'])} approved | "
      f"{len([c for c in GENERATED_CELLS if c['status']=='flagged'])} flagged")
print()
print("─" * 60)
print("NOTEBOOK PREVIEW (first 80 lines):")
print("─" * 60)
print("\n".join(notebook_text.splitlines()[:80]))

---
## Cell 12 — Inspect Individual Generated Cells
Use `show_cell(step_id)` to print the full generated code for any cell.

In [ ]:
def show_cell(step_id: int):
    """Print the full generated cell code for a given step ID."""
    for cell in GENERATED_CELLS:
        if cell["step_id"] == step_id:
            status_icon = "✅" if cell["status"] == "approved" else "⛔"
            print(f"{status_icon} Cell {step_id} — {cell['status'].upper()}")
            if cell.get("flag_reason"):
                print(f"Flag reason: {cell['flag_reason']}")
            print("─" * 60)
            print(cell["cell_code"])
            print("─" * 60)
            return
    print(f"❌ No cell found for step_id={step_id}")


def show_all_cells():
    """Print a summary table of all generated cells."""
    print(f"{'Step':>5}  {'Status':12}  {'Type':15}  {'Instruction (truncated)'}")
    print("─" * 90)
    for cell in GENERATED_CELLS:
        step = next((s for s in APPROVED_PLAN["plan_steps"] if s["step_id"] == cell["step_id"]), {})
        icon = "✅" if cell["status"] == "approved" else "⛔"
        print(f"{cell['step_id']:>5}  {icon} {cell['status']:10}  "
              f"{step.get('step_type','?'):15}  "
              f"{step.get('instruction','')[:50]}...")


# Show summary table
show_all_cells()
print()
print("Use: show_cell(step_id=1) to print any cell's full code")